In [ ]:
#Install the required libraries

In [1]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn pillow opencv-python-headless ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Import the libraries

In [3]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from PIL import Image

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("AutoClean AI: Libraries imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("OpenCV version:", cv2.__version__)

AutoClean AI: Libraries imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
OpenCV version: 4.11.0


In [ ]:
#Create the project folders

In [5]:
PROJECT_DIR = Path.cwd() / "AutoClean_AI"

folders = [
    "data/clean",
    "data/dirty",
    "data/reference_images",
    "outputs/cleaned",
    "outputs/reports",
    "outputs/repair_history",
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR.resolve())
print("\nFolder structure:")

for folder in folders:
    print(f"  {folder}/")

Project folder: C:\Users\tsame\ML_Project\AutoClean_AI

Folder structure:
  data/clean/
  data/dirty/
  data/reference_images/
  outputs/cleaned/
  outputs/reports/
  outputs/repair_history/


In [ ]:
#Load a real tabular dataset

In [7]:
from ucimlrepo import fetch_ucirepo

# Fetch the real dataset from the UCI repository
dataset = fetch_ucirepo(id=17)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# Combine features and target for initial inspection
df_original = X.copy()
df_original["diagnosis"] = y.iloc[:, 0].values

print("Dataset loaded successfully!")
print("Number of rows:", df_original.shape[0])
print("Number of columns:", df_original.shape[1])

display(df_original.head())

Dataset loaded successfully!
Number of rows: 569
Number of columns: 31


,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


In [ ]:
#Inspect the original data quality

In [9]:
print("Dataset dimensions:", df_original.shape)

print("\nColumn data types:")
display(df_original.dtypes.to_frame("Data Type"))

print("\nMissing values per column:")
display(
    df_original.isnull()
    .sum()
    .to_frame("Missing Values")
    .sort_values("Missing Values", ascending=False)
)

print("\nDuplicate rows:", df_original.duplicated().sum())

print("\nDiagnosis distribution:")
display(df_original["diagnosis"].value_counts(dropna=False))

Dataset dimensions: (569, 31)

Column data types:


,Data Type
radius1,float64
texture1,float64
perimeter1,float64
area1,float64
smoothness1,float64
compactness1,float64
concavity1,float64
concave_points1,float64
symmetry1,float64
fractal_dimension1,float64



Missing values per column:


,Missing Values
radius1,0
concavity2,0
fractal_dimension3,0
symmetry3,0
concave_points3,0
concavity3,0
compactness3,0
smoothness3,0
area3,0
perimeter3,0



Duplicate rows: 0

Diagnosis distribution:


diagnosis
B    357
M    212
Name: count, dtype: int64

In [ ]:
#prepare the image dataset

In [11]:
%pip install -q tensorflow

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 7.36.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [12]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

# Load the real CIFAR-10 dataset
(x_train_img, y_train_img), (x_test_img, y_test_img) = (
    tf.keras.datasets.cifar10.load_data()
)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", x_train_img.shape)
print("Testing images:", x_test_img.shape)
print("Training labels:", y_train_img.shape)

TensorFlow version: 2.21.0
 46768128/170498071 ━━━━━━━━━━━━━━━━━━━━ 40:35 20us/step

Exception: URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: None -- retrieval incomplete: got only 46784000 out of 170498071 bytes

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_img[i])
    plt.title(class_names[int(y_train_img[i, 0])])
    plt.axis("off")

plt.tight_layout()
plt.show()